In [ ]:
import numpy as np
from datetime import date, timedelta

start_date = date(2026, 6, 1)
end_date = date(2026, 6, 7)


In [ ]:
ssts = []
for i in range((end_date - start_date).days):
    current_date = start_date + timedelta(days=i)
    for day in (0, 12):
        day_str = str(day).zfill(2)
        sst_example = np.load(f"../data/{current_date.strftime('%m/%d')}/{day_str}/sst.npy", allow_pickle=True)
        ssts.append(sst_example.astype(np.float64))

sst_full = np.stack(ssts, axis=-1)

In [ ]:
# Mirrors the ocean-mask construction in src/weather/model.py's
# `_mask_and_stack` (used by WeatherModel.fit): a cell counts as "ocean"
# only if it is non-NaN across every training snapshot, for every modeled
# variable (here just `sst`). Unlike the production pipeline — which then
# ravels (lat, lon) into a single `ocean_cell` Tucker mode and drops land
# cells from the tensor entirely — this notebook keeps `sst_full`'s 3D
# (lat, lon, time) shape so lat and lon stay separate Tucker modes to
# explore. Land/ice cells are instead flagged via a mask passed to
# `partial_tucker(mask=...)`, which iteratively imputes them from the
# current low-rank reconstruction instead of contributing raw NaN (or an
# arbitrary sentinel like -9999) to the decomposition.
ocean_mask = ~np.isnan(sst_full).any(axis=-1)  # (1121, 2880) bool

# partial_tucker's `mask` must match `tensor`'s shape, with 1.0 for
# observed entries and 0.0 for entries to impute.
_mask = np.broadcast_to(ocean_mask[:, :, None], sst_full.shape).astype(np.float64)

# The masked HOI loop overwrites entries where tucker_mask == 0 on its
# first iteration, but tensorly computes the tensor's norm once up front
# using these raw values — NaN would poison that norm, and a sentinel like
# -9999 would inflate it. 0.0 is a neutral placeholder immediately
# replaced by the reconstruction-based imputation.
sst_filled = np.where(ocean_mask[:, :, None], sst_full, 0.0)